# ACRS on AppWorld: O1–O4 + G1 with open models on a free or Colab Pro GPU

Jenisha T · M S Ramaiah University of Applied Sciences · 24ETRP720001

`Runtime → Run all`. The notebook:
1. picks the largest open models that fit this GPU;
2. serves them with vLLM;
3. runs the official-style baseline, full ACRS and leave-one-out ablations on AppWorld **dev**;
4. scores everything with AppWorld's official evaluator (TGC/SGC) and runs paired tests.

**Rules (AppWorld):** tune on train/dev only; test splits run once per final configuration. Do not publish unencrypted AppWorld task or API content. Experiment outputs stay in this runtime; only aggregate summaries leave it.

In [ ]:
import os, subprocess, json, time, sys
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
WORK = '/kaggle/working' if IN_KAGGLE else '/content'
os.chdir(WORK)
mem = int(subprocess.run(['nvidia-smi','--query-gpu=memory.total','--format=csv,noheader,nounits'],capture_output=True,text=True).stdout.split()[0])
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True).stdout.strip()
if mem >= 39000:   LARGE, SMALL, UL, US, LEN = 'Qwen/Qwen3-32B-AWQ', 'Qwen/Qwen3-4B-AWQ', 0.62, 0.22, 24576
elif mem >= 22000: LARGE, SMALL, UL, US, LEN = 'Qwen/Qwen3-14B-AWQ', 'Qwen/Qwen3-4B-AWQ', 0.58, 0.28, 16384
# T4 (16 GB, compute 7.5): 8B-AWQ weights take ~5.7 GB, so the large tier needs ~2/3 of memory for a usable
# KV cache; eager mode frees the CUDA-graph reserve, and the small tier is the 0.6B model.
else: LARGE, SMALL, UL, US, LEN = 'Qwen/Qwen3-8B-AWQ', 'Qwen/Qwen3-0.6B', 0.66, 0.26, 10240
EAGER = mem < 22000
# Colab's T4 VM has ~12.7 GB RAM; each AppWorld worker holds its own app databases, so keep few workers there.
WORKERS = 2 if EAGER else 8
# Kaggle 2xT4 (30 GB RAM): one server per GPU, a stronger small tier, more workers.
NGPU = len(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout.strip().splitlines())
if NGPU >= 2 and EAGER:
    LARGE, SMALL, UL, US, LEN, WORKERS = 'Qwen/Qwen3-8B-AWQ', 'Qwen/Qwen3-4B-AWQ', 0.90, 0.90, 16384, 4
print(NGPU, 'x', gpu, mem, 'MiB ->', LARGE, '+', SMALL)

In [ ]:
os.chdir(WORK)
!rm -rf acrs-appworld && git clone -q --depth 1 https://github.com/joyjeni/acrs-appworld.git
%cd acrs-appworld
!git log --oneline -1
# vLLM (pydantic 2) and AppWorld 0.1.3 (pydantic 1) cannot share an environment:
# each gets its own venv; they talk over HTTP only.
!pip -q install uv 2>&1 | tail -1
!uv venv -q --clear --python 3.12 /opt/vllm && uv pip install -q --python /opt/vllm/bin/python vllm 2>&1 | tail -2
!/opt/vllm/bin/python -c "import vllm; print('vllm', vllm.__version__)"
!uv venv -q --clear --python 3.11 /opt/aw && uv pip install -q --python /opt/aw/bin/python "appworld==0.1.3.post1" numpy scipy pytest 2>&1 | tail -2
!/opt/aw/bin/python -c "import appworld; print('appworld', appworld.__version__)"
# AppWorld data gets its own root: `download data` would otherwise overwrite ./data (the ToolBench bundle).
os.environ['APPWORLD_ROOT'] = AW_ROOT = f'{WORK}/appworld_root'; os.makedirs(AW_ROOT, exist_ok=True)
!/opt/aw/bin/appworld install 2>&1 | tail -1
!cd $AW_ROOT && /opt/aw/bin/appworld download data 2>&1 | tail -1
!/opt/aw/bin/python -m pytest -q tests 2>&1 | tail -2


In [ ]:
def serve(model, port, util, maxlen, gpu_id=None):
    log = open(f'/tmp/vllm_{port}.log', 'w')
    env = dict(os.environ, **({'CUDA_VISIBLE_DEVICES': str(gpu_id)} if gpu_id is not None else {}))
    return subprocess.Popen(['/opt/vllm/bin/python','-m','vllm.entrypoints.openai.api_server','--model',model,'--port',str(port),
        '--gpu-memory-utilization',str(util),'--max-model-len',str(maxlen),'--disable-log-requests', '--dtype', 'half' if EAGER else 'auto'] + (['--enforce-eager'] if EAGER else []), stdout=log, stderr=log, env=env)

def wait(port, proc, timeout=1800):
    import urllib.request
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None: break  # server died: fail fast with its log
        try:
            urllib.request.urlopen(f'http://localhost:{port}/v1/models', timeout=5); return True
        except Exception: time.sleep(10)
    raise RuntimeError(open(f'/tmp/vllm_{port}.log').read()[-3000:])

p_large = serve(LARGE, 8001, UL, LEN, 0 if NGPU >= 2 else None); wait(8001, p_large)
p_small = serve(SMALL, 8002, US, LEN, 1 if NGPU >= 2 else None); wait(8002, p_small)
L = f'{LARGE}@http://localhost:8001/v1'; S = f'{SMALL}@http://localhost:8002/v1'
print('servers up')

## 1 · Smoke test: 5 dev tasks, baseline vs full (about 10 min)

In [ ]:
for cfg in ['baseline', 'full']:
    !/opt/aw/bin/python experiments/appworld_run.py --split dev --config {cfg} --limit 5 --workers {WORKERS} --tag smoke --small "{S}" --large "{L}" 2>&1 | tail -25

## 2 · Dev pilot: baseline, full ACRS and leave-one-out ablations

In [ ]:
CONFIGS = ['baseline', 'full', 'no_o1', 'no_o2', 'no_o3', 'no_o4', 'no_g1']
for cfg in CONFIGS:
    t0 = time.time()
    !/opt/aw/bin/python experiments/appworld_run.py --split dev --config {cfg} --workers {WORKERS} --small "{S}" --large "{L}" > /tmp/run_{cfg}.log 2>&1
    print(cfg, f'{(time.time()-t0)/60:.1f} min'); print(open(f'/tmp/run_{cfg}.log').read()[-600:])

## 3 · Results and paired significance (McNemar exact test on per-task success, bootstrap CI)

In [ ]:
import glob, random, math
import pandas as pd
from scipy.stats import binomtest
R = 'experiments/results/appworld'
def load(cfg):
    s = json.load(open(f'{R}/acrs_{cfg}_s0_dev.summary.json'))
    ev = json.load(open(f'{R}/acrs_{cfg}_s0_dev.evaluation.json'))
    per = {t: v.get('success', False) for t, v in ev.get('individual', {}).items()}
    return s, per
rows, per = [], {}
for cfg in CONFIGS:
    try: s, p = load(cfg)
    except FileNotFoundError: continue
    per[cfg] = p; a = s['official']
    rows.append({'config': cfg, 'TGC': a.get('task_goal_completion'), 'SGC': a.get('scenario_goal_completion'),
                 'tokens(M)': round(s['tokens']/1e6, 2), 'small/large': f"{s['router']['small']}/{s['router']['large']}",
                 'certified/abstain': f"{s['consensus']['certified']}/{s['consensus']['abstain_disagree']+s['consensus']['abstain_all_failed']}",
                 'policy denials': s['policyDenials'], 'fidelity': round(s['meanFidelity'], 3), 'failed eps': s['episodesFailed']})
display(pd.DataFrame(rows))
def mcnemar(a, b):
    ks = sorted(set(per[a]) & set(per[b])); b01 = sum(per[a][k] and not per[b][k] for k in ks); b10 = sum(per[b][k] and not per[a][k] for k in ks)
    return b01, b10, (binomtest(b01, b01+b10, 0.5).pvalue if b01+b10 else 1.0)
def boot(a, b, n=5000):
    ks = sorted(set(per[a]) & set(per[b])); d = [int(per[a][k]) - int(per[b][k]) for k in ks]; rng = random.Random(0)
    m = sorted(sum(rng.choice(d) for _ in d)/len(d) for _ in range(n)); return round(m[int(.025*n)], 4), round(m[int(.975*n)], 4)
for other in [c for c in per if c != 'full']:
    w, l, p = mcnemar('full', other)
    print(f'full vs {other:9s}: full-only wins {w}, other-only wins {l}, McNemar p={p:.4f}, TGC diff 95% CI {boot("full", other)}')

In [ ]:
import shutil
shutil.make_archive(f'{WORK}/acrs_appworld_summaries', 'zip', R)
print('download:', f'{WORK}/acrs_appworld_summaries.zip')